# 08 — PPO with Symbolic (RAM) Observations

PPO agent using the RAM-based symbolic grid (13x16) instead of pixels.

**Setup:**
- Observation: flattened 13x16 grid with 4 frame stack = 832-dim vector
- Policy: MlpPolicy with [512, 512] hidden layers
- Much faster than pixel-based (no CNN, no image processing)
- Same PPO hyperparameters as pixel version

In [1]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

/home/contente/Documents/ENSTA/autonomous/CSC-52081-ContinousMountainCar


In [2]:
import torch
from stable_baselines3 import PPO

from src.wrappers import make_symbolic_vec_env, make_symbolic_env
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import PPOConfig

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using CUDA: True
GPU: NVIDIA GeForce GTX 1650


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [ ]:
# Create 8 parallel symbolic environments
NUM_ENVS = 8

env = make_symbolic_vec_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4,
    n_stack=4,
    flatten=True,
    num_envs=NUM_ENVS,
)
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')
print(f'Parallel envs: {NUM_ENVS}')

In [ ]:
# Phase 1: Train from scratch
config = PPOConfig(
    policy='MlpPolicy',
    learning_rate=2.5e-4,
    n_steps=512,
    batch_size=256,
    n_epochs=4,
    gamma=0.9,           # vietnh1009 uses 0.9
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    total_timesteps=2_000_000,
)

model = PPO(
    policy=config.policy,
    env=env,
    learning_rate=config.learning_rate,
    n_steps=config.n_steps,
    batch_size=config.batch_size,
    n_epochs=config.n_epochs,
    gamma=config.gamma,
    gae_lambda=config.gae_lambda,
    clip_range=config.clip_range,
    ent_coef=config.ent_coef,
    vf_coef=config.vf_coef,
    max_grad_norm=config.max_grad_norm,
    policy_kwargs=dict(net_arch=[512, 512]),
    tensorboard_log='../logs/symbolic_ppo',
    verbose=1,
    device='cpu',  # MLP is faster on CPU
)

print(f'Phase 1: {config.total_timesteps:,} timesteps')
print(f'Device: {model.device}')
print(f'Batch per rollout: {config.n_steps * NUM_ENVS} samples')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../logs/symbolic_ppo

In [ ]:
# Phase 1: Train for 2M steps
callback = CheckpointAndLogCallback(
    save_path='../models/symbolic_ppo',
    save_freq=50_000,
)

model.learn(
    total_timesteps=config.total_timesteps,
    callback=callback,
    log_interval=1,
)
model.save('../models/symbolic_ppo/phase1_model')
print('Phase 1 complete!')

In [ ]:
# Phase 2: Load best model, lower lr, train 2M more
from stable_baselines3 import PPO

model = PPO.load('../models/symbolic_ppo/phase1_model', env=env, device='cpu')
model.learning_rate = 1e-5

callback2 = CheckpointAndLogCallback(
    save_path='../models/symbolic_ppo',
    save_freq=50_000,
)

model.learn(
    total_timesteps=2_000_000,
    callback=callback2,
    log_interval=1,
)
model.save('../models/symbolic_ppo/final_model')
print('Phase 2 complete!')

In [ ]:
# Plot training results
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

rewards = callback.episode_rewards + (callback2.episode_rewards if 'callback2' in dir() else [])
lengths = callback.episode_lengths + (callback2.episode_lengths if 'callback2' in dir() else [])
flags = callback.episode_flags + (callback2.episode_flags if 'callback2' in dir() else [])

if len(rewards) > 0:
    window = min(100, len(rewards))

    ax = axes[0]
    ax.plot(rewards, alpha=0.3, color='blue')
    if len(rewards) > window:
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), smoothed, color='blue', linewidth=2)
    ax.set_title('Episode Rewards')
    ax.set_xlabel('Episode')

    ax = axes[1]
    ax.plot(lengths, alpha=0.3, color='orange')
    if len(lengths) > window:
        smoothed = np.convolve(lengths, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(lengths)), smoothed, color='orange', linewidth=2)
    ax.set_title('Episode Lengths')
    ax.set_xlabel('Episode')

    ax = axes[2]
    if len(flags) > 0:
        cumulative_flags = np.cumsum(flags) / (np.arange(len(flags)) + 1)
        ax.plot(cumulative_flags, color='green', linewidth=2)
    ax.set_title('Cumulative Flag Rate')
    ax.set_xlabel('Episode')
    ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('../models/symbolic_ppo/training_curves.png', dpi=150)
    plt.show()
else:
    print('No episode data collected yet.')

In [ ]:
# Evaluate final model
from stable_baselines3 import PPO
import numpy as np

eval_model = PPO.load('../models/symbolic_ppo/final_model')

eval_env = make_symbolic_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4, n_stack=4, flatten=True,
)

rewards = []
lengths = []
flags = []

for ep in range(10):
    reset_result = eval_env.reset()
    obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result
    done = False
    total_reward = 0.0
    steps = 0
    flag = False

    while not done:
        action, _ = eval_model.predict(obs, deterministic=True)
        step_result = eval_env.step(int(action))
        if len(step_result) == 5:
            obs, reward, terminated, truncated, info = step_result
            done = terminated or truncated
        else:
            obs, reward, done, info = step_result
        total_reward += float(reward)
        steps += 1
        if isinstance(info, dict) and info.get('flag_get', False):
            flag = True

    rewards.append(total_reward)
    lengths.append(steps)
    flags.append(flag)
    status = 'FLAG!' if flag else 'DEAD'
    print(f'Episode {ep+1}: reward={total_reward:.1f}, steps={steps}, {status}')

print(f'\nMean reward: {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}')
print(f'Mean length: {np.mean(lengths):.0f}')
print(f'Flag rate: {np.mean(flags):.0%}')

eval_env.close()

## Transfer Learning: World 1-2

Fine-tune the 1-1 trained model on level 1-2.

In [3]:
# Create env for world 1-2
env_12 = make_symbolic_vec_env(
    env_id='SuperMarioBros-1-2-v3',
    skip=4,
    n_stack=4,
    flatten=True,
    num_envs=8,
)
print(f'Observation space: {env_12.observation_space.shape}')
print(f'Action space: {env_12.action_space.n}')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Observation space: (832,)
Action space: 7


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. 

In [4]:
# Load 1-1 model and fine-tune on 1-2
from stable_baselines3 import PPO

model_12 = PPO.load(
    '../models/symbolic_ppo/final_model',
    env=env_12,
    device='cpu',
    learning_rate=2.5e-4,  # override saved lr
)

callback_12 = CheckpointAndLogCallback(
    save_path='../models/symbolic_ppo_1_2',
    save_freq=50_000,
)

print(f'LR: {model_12.learning_rate}')
print('Training on World 1-2 (transfer from 1-1)...')
model_12.learn(
    total_timesteps=2_000_000,
    callback=callback_12,
    log_interval=1,
    tb_log_name='PPO_1_2',
)
model_12.save('../models/symbolic_ppo_1_2/final_model')
print('World 1-2 training complete!')

LR: 0.00025
Training on World 1-2 (transfer from 1-1)...
Logging to ../logs/symbolic_ppo/PPO_1_2_2


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional infor

-----------------------------
| time/              |      |
|    fps             | 238  |
|    iterations      | 1    |
|    time_elapsed    | 17   |
|    total_timesteps | 4096 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 227         |
|    iterations           | 2           |
|    time_elapsed         | 35          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.024773296 |
|    clip_fraction        | 0.178       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.861      |
|    explained_variance   | -0.365      |
|    learning_rate        | 0.00025     |
|    loss                 | 0.881       |
|    n_updates            | 3916        |
|    policy_gradient_loss | -0.00264    |
|    value_loss           | 3.85        |
-----------------------------------------
----------------------------------

/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


----------------------------------------
| time/                   |            |
|    fps                  | 272        |
|    iterations           | 78         |
|    time_elapsed         | 1172       |
|    total_timesteps      | 319488     |
| train/                  |            |
|    approx_kl            | 0.01825167 |
|    clip_fraction        | 0.123      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.755     |
|    explained_variance   | 0.752      |
|    learning_rate        | 0.00025    |
|    loss                 | 0.515      |
|    n_updates            | 4220       |
|    policy_gradient_loss | -0.0112    |
|    value_loss           | 1.6        |
----------------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 272         |
|    iterations           | 79          |
|    time_elapsed         | 1186        |
|    total_timesteps      | 323584      |
| train/  

...

-----------------------------------------
| time/                   |             |
|    fps                  | 281         |
|    iterations           | 103         |
|    time_elapsed         | 1501        |
|    total_timesteps      | 421888      |
| train/                  |             |
|    approx_kl            | 0.023092836 |
|    clip_fraction        | 0.204       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.968      |
|    explained_variance   | 0.927       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.289       |
|    n_updates            | 4320        |
|    policy_gradient_loss | -0.00937    |
|    value_loss           | 0.88        |
-----------------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 281         |
|    iterations           | 104         |
|    time_elapsed         | 1512        |
|    total_timesteps      | 425984

/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


----------------------------------------
| time/                   |            |
|    fps                  | 288        |
|    iterations           | 123        |
|    time_elapsed         | 1743       |
|    total_timesteps      | 503808     |
| train/                  |            |
|    approx_kl            | 0.03106212 |
|    clip_fraction        | 0.277      |
|    clip_range           | 0.2        |
|    entropy_loss         | -0.992     |
|    explained_variance   | 0.93       |
|    learning_rate        | 0.00025    |
|    loss                 | 0.264      |
|    n_updates            | 4400       |
|    policy_gradient_loss | -0.0142    |
|    value_loss           | 0.835      |
----------------------------------------
----------------------------------------
| time/                   |            |
|    fps                  | 289        |
|    iterations           | 124        |
|    time_elapsed         | 1756       |
|    total_timesteps      | 507904     |
| train/        

/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


-----------------------------------------
| time/                   |             |
|    fps                  | 290         |
|    iterations           | 133         |
|    time_elapsed         | 1878        |
|    total_timesteps      | 544768      |
| train/                  |             |
|    approx_kl            | 0.030682819 |
|    clip_fraction        | 0.2         |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.03       |
|    explained_variance   | 0.966       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.0731      |
|    n_updates            | 4440        |
|    policy_gradient_loss | -0.0161     |
|    value_loss           | 0.408       |
-----------------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 289         |
|    iterations           | 134         |
|    time_elapsed         | 1894        |
|    total_timesteps      | 548864

In [ ]:
# Watch: python watch_agent.py --model ../models/symbolic_ppo_1_2/final_model --env SuperMarioBros-1-2-v3